# Ứng dụng Học sâu trong Phát hiện Tấn công DDoS trên IoT
**Dữ liệu thật:** CIC-IoT-2023 (UNB/CIC) và N-BaIoT (UCI).

Notebook này chạy đúng pipeline trong `src/`: kiểm tra dữ liệu → lấy mẫu & làm sạch → EDA → tiền xử lý → 5 baseline ML vs 4 mô hình DL → đánh giá trên tập test → xuất ONNX.
Trước khi chạy: đặt dữ liệu vào `datasets/` theo `HUONG_DAN_LAM_BAI.md` (mục 2).

In [ ]:
import os, sys, json
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'src'))
import pandas as pd
from IPython.display import Image, display, Markdown
print('Project root:', ROOT)

## 1. Kiểm tra dữ liệu đã tải (Data check)

In [ ]:
from check_data import check_cic, check_nbaiot
reports = [check_cic(os.path.join(ROOT, 'datasets', 'CIC-IoT-2023')),
           check_nbaiot(os.path.join(ROOT, 'datasets', 'N-BaIoT'))]
for r in reports:
    r.show()
assert sum(r.n_errors for r in reports) == 0, 'Sửa các lỗi ERROR ở trên trước khi chạy tiếp'

## 2. Chạy thí nghiệm cho từng bộ dữ liệu (EDA → train → evaluate)
Nếu đã có `results/<dataset>/results.json` thì notebook đọc lại kết quả, không huấn luyện lại (đặt `RETRAIN = True` để chạy lại).

In [ ]:
from run_experiments import run_dataset, default_args
RETRAIN = False
DATASETS = ['cic_iot2023', 'n_baiot']
summaries = {}
for name in DATASETS:
    path = os.path.join(ROOT, 'results', name, 'results.json')
    if os.path.exists(path) and not RETRAIN:
        summaries[name] = json.load(open(path, encoding='utf-8'))
        print(f'{name}: loaded existing results')
    else:
        summaries[name] = run_dataset(name, default_args())

## 3. EDA – phân bố lớp, tương quan, đặc trưng quan trọng

In [ ]:
for name in DATASETS:
    d = summaries[name]
    display(Markdown(f"### {d['dataset']}  ({d['eda']['n_rows']:,} dòng, {d['eda']['n_input_features']} đặc trưng gốc → {len(d['features_selected'])} đặc trưng chọn)"))
    print('Làm sạch:', d['eda']['cleaning'])
    print('Cân bằng tập train:', d['balancing']['before'], '->', d['balancing']['after'])
    for fig in ['class_distribution.png', 'correlation_heatmap.png', 'feature_importance.png', 'feature_distributions.png']:
        display(Image(os.path.join(ROOT, 'results', name, 'figures', fig)))

## 4. Bảng so sánh Baseline vs Deep Learning (tập test)

In [ ]:
cols = ['Model','Category','Accuracy','Precision_macro','Recall_macro','F1_macro','ROC_AUC_macro','Latency_ms_single','Throughput_per_s','Size_KB','selected_params']
for name in DATASETS:
    d = summaries[name]
    display(Markdown(f"### {d['dataset']}"))
    display(pd.DataFrame(d['results'])[cols].sort_values('F1_macro', ascending=False).reset_index(drop=True))
    r = d['requirement_5']
    display(Markdown(f"**Yêu cầu #5:** DL tốt nhất **{r['best_deep']}** F1 = {r['best_deep_f1']:.3f}% vs baseline tốt nhất **{r['best_baseline']}** F1 = {r['best_baseline_f1']:.3f}% → chênh {r['gap_f1']:+.3f} → **{'ĐẠT' if r['passed'] else 'CHƯA ĐẠT'}**"))
    display(Image(os.path.join(ROOT, 'results', name, 'figures', 'model_comparison.png')))

## 5. Ma trận nhầm lẫn & đường cong huấn luyện DL

In [ ]:
for name in DATASETS:
    for fig in ['cm_best_baseline.png', 'cm_best_dl.png', 'dl_training_curves.png']:
        display(Image(os.path.join(ROOT, 'results', name, 'figures', fig)))

## 6. Mô hình triển khai (ONNX + INT8) cho Kali

In [ ]:
for name in DATASETS:
    print(name, json.dumps(summaries[name]['onnx'], indent=2))
print(open(os.path.join(ROOT, 'deploy_kali', 'model', 'MODEL_INFO.json'), encoding='utf-8').read())

## 7. Đánh giá phát hiện trên file pcap ghi trong lab (tùy chọn)
Đặt các file `.pcap` ghi bằng tcpdump trên Kali vào thư mục `captures/` (xem hướng dẫn mục 5).

In [ ]:
import glob
from realtime.detection_engine import IoTRealtimeMonitor, default_bundle_paths
pcaps = sorted(glob.glob(os.path.join(ROOT, 'captures', '*.pcap*')))
if not pcaps:
    print('Chưa có file pcap trong captures/')
model_path, prep_path = default_bundle_paths(ROOT)
rows = []
for p in pcaps:
    mon = IoTRealtimeMonitor(model_path, prep_path, log_file=os.devnull)
    s = mon.replay(p, report_path=p + '.report.json')
    rows.append({k: v for k, v in s.items() if k != 'per_window'})
pd.DataFrame(rows)